In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [12]:
df = pd.read_csv("clean_dataset.csv")

df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'clean_dataset.csv'

In [ ]:
df = pd.read_csv("retail-orders-raw.csv")

df.head()

,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10.0,Paid
1,RT-1002,03/01/2026,Fresher,Bengaluru,Course Access,1,1499,0.0,paid
2,RT-1003,2026-01-05,student,Chennai,Course Access,1,1499,NaN,Pending
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid


In [13]:
print("Number of Rows:", df.shape[0])
print("Number of Columns:", df.shape[1])

df.info()

Number of Rows: 12
Number of Columns: 9
<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          12 non-null     str    
 1   order_date        11 non-null     str    
 2   customer_segment  12 non-null     str    
 3   city              11 non-null     str    
 4   category          12 non-null     str    
 5   quantity          12 non-null     str    
 6   unit_price        12 non-null     int64  
 7   discount_pct      11 non-null     float64
 8   payment_status    12 non-null     str    
dtypes: float64(1), int64(1), str(7)
memory usage: 1.6 KB


In [14]:
# Check missing values
missing_values = df.isnull().sum()

print("Missing Values in Each Column:")
print(missing_values)

Missing Values in Each Column:
order_id            0
order_date          1
customer_segment    0
city                1
category            0
quantity            0
unit_price          0
discount_pct        1
payment_status      0
dtype: int64


In [15]:
# Check duplicate rows
duplicates = df.duplicated().sum()

print("Number of Duplicate Rows:", duplicates)

Number of Duplicate Rows: 1


In [16]:
# Check data types
print("Data Types of Each Column:")
print(df.dtypes)

Data Types of Each Column:
order_id                str
order_date              str
customer_segment        str
city                    str
category                str
quantity                str
unit_price            int64
discount_pct        float64
payment_status          str
dtype: object


In [17]:
# Check unique values
for column in df.columns:
    print("\n", column)
    print(df[column].unique())


 order_id
<ArrowStringArray>
['RT-1001', 'RT-1002', 'RT-1003', 'RT-1004', 'RT-1005', 'RT-1006', 'RT-1007',
 'RT-1008', 'RT-1009', 'RT-1010', 'RT-1011']
Length: 11, dtype: str

 order_date
<ArrowStringArray>
['2026-01-03', '03/01/2026', '2026-01-05', '2026-01-07', '2026-01-09',
 '2026-13-10', '2026-01-12', '2026-01-14', '2026-01-16', '2026-01-18',
          nan]
Length: 11, dtype: str

 customer_segment
<ArrowStringArray>
['Student', 'Fresher', 'student', 'Professional']
Length: 4, dtype: str

 city
<ArrowStringArray>
['Chennai', 'Bengaluru', 'Hyderabad', nan, 'Pune', 'Mumbai', 'Delhi', 'Kochi']
Length: 8, dtype: str

 category
<ArrowStringArray>
['Learning Kit', 'Course Access', 'Mentor Session']
Length: 3, dtype: str

 quantity
<ArrowStringArray>
['2', '1', '3', '-1', 'two']
Length: 5, dtype: str

 unit_price
[ 799 1499  999]

 discount_pct
[ 10.   0.  nan   5. 105.  15.]

 payment_status
<ArrowStringArray>
['Paid', 'paid', 'Pending', 'Failed', 'Refunded']
Length: 5, dtype: str


In [18]:
# Convert date and quantity columns

df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")

print(df.dtypes)

order_id                       str
order_date          datetime64[us]
customer_segment               str
city                           str
category                       str
quantity                   float64
unit_price                   int64
discount_pct               float64
payment_status                 str
dtype: object


In [19]:
# Handle missing values

df["city"] = df["city"].fillna("Unknown")
df["discount_pct"] = df["discount_pct"].fillna(0)

print("Missing values after cleaning:")
print(df.isnull().sum())

Missing values after cleaning:
order_id            0
order_date          3
customer_segment    0
city                0
category            0
quantity            1
unit_price          0
discount_pct        0
payment_status      0
dtype: int64


In [20]:
# Reload original dataset and correctly handle mixed date formats

df = pd.read_csv("retail-orders-raw.csv")

df["order_date"] = pd.to_datetime(
    df["order_date"],
    format="mixed",
    errors="coerce"
)

df["quantity"] = pd.to_numeric(
    df["quantity"],
    errors="coerce"
)

df["city"] = df["city"].fillna("Unknown")
df["discount_pct"] = df["discount_pct"].fillna(0)

print("Missing values after correction:")
print(df.isnull().sum())

Missing values after correction:
order_id            0
order_date          2
customer_segment    0
city                0
category            0
quantity            1
unit_price          0
discount_pct        0
payment_status      0
dtype: int64


In [21]:
# Find rows with missing date or quantity

problem_rows = df[df["order_date"].isna() | df["quantity"].isna()]

print("Rows with missing date or quantity:")
print(problem_rows)

Rows with missing date or quantity:
   order_id order_date customer_segment       city        category  quantity  \
6   RT-1006        NaT          Student       Pune    Learning Kit      -1.0   
8   RT-1008 2026-01-14          Fresher  Bengaluru   Course Access       NaN   
11  RT-1011        NaT          Student      Kochi  Mentor Session       1.0   

    unit_price  discount_pct payment_status  
6          799          10.0           Paid  
8         1499           0.0        Pending  
11         999           0.0           Paid  


In [22]:
# Check the original values of problematic rows

raw_df = pd.read_csv("retail-orders-raw.csv")

problem_index = df[df["order_date"].isna() | df["quantity"].isna()].index

print("Problematic rows from original dataset:")
print(raw_df.loc[problem_index])

Problematic rows from original dataset:
   order_id  order_date customer_segment       city        category quantity  \
6   RT-1006  2026-13-10          Student       Pune    Learning Kit       -1   
8   RT-1008  2026-01-14          Fresher  Bengaluru   Course Access      two   
11  RT-1011         NaN          Student      Kochi  Mentor Session        1   

    unit_price  discount_pct payment_status  
6          799          10.0           Paid  
8         1499           0.0        Pending  
11         999           0.0           Paid  


In [23]:
raw_df = pd.read_csv("retail-orders-raw.csv")

problem_index = df[df["order_date"].isna() | df["quantity"].isna()].index

print("Problematic rows from original dataset:")
print(raw_df.loc[problem_index])

Problematic rows from original dataset:
   order_id  order_date customer_segment       city        category quantity  \
6   RT-1006  2026-13-10          Student       Pune    Learning Kit       -1   
8   RT-1008  2026-01-14          Fresher  Bengaluru   Course Access      two   
11  RT-1011         NaN          Student      Kochi  Mentor Session        1   

    unit_price  discount_pct payment_status  
6          799          10.0           Paid  
8         1499           0.0        Pending  
11         999           0.0           Paid  


In [24]:
# Remove rows with missing important values
df = df.dropna(subset=["order_date", "quantity"])

# Remove duplicate rows
df = df.drop_duplicates()

# Reset index
df = df.reset_index(drop=True)

print("Rows after cleaning:", len(df))
print("Missing values:")
print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())

Rows after cleaning: 8
Missing values:
order_id            0
order_date          0
customer_segment    0
city                0
category            0
quantity            0
unit_price          0
discount_pct        0
payment_status      0
dtype: int64
Duplicate rows: 0


In [25]:
# Save cleaned dataset

df.to_csv("clean_dataset.csv", index=False)

print("Cleaned dataset saved successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Cleaned dataset saved successfully!
Rows: 8
Columns: 9
